In [1]:
import os
from pathlib import Path
import pandas as pd
from collections import Counter, defaultdict

possible_bases = [
    Path("/kaggle/input/h-and-m-personalized-fashion-recommendations"),
    Path("./kaggle/input/h-and-m-personalized-fashion-recommendations"),
    Path("../input/h-and-m-personalized-fashion-recommendations"),
]
base_path = None
for p in possible_bases:
    if p.is_dir():
        base_path = p
        break
if base_path is None:
    raise FileNotFoundError(
        "Competition data directory not found in expected locations."
    )

sample_path = base_path / "sample_submission.csv"
train_path = base_path / "transactions_train.csv"

sample_sub = pd.read_csv(sample_path, dtype={"customer_id": str})
sample_sub = sample_sub.sort_values("customer_id").reset_index(drop=True)



In [2]:
chunksize = 1_000_000
max_date = None

# First pass: determine the most recent transaction date
for chunk in pd.read_csv(
    train_path,
    usecols=["t_dat"],
    dtype={"t_dat": str},
    chunksize=chunksize,
):
    chunk["t_dat"] = pd.to_datetime(chunk["t_dat"])
    cur_max = chunk["t_dat"].max()
    if max_date is None or cur_max > max_date:
        max_date = cur_max

# Define recency windows
cutoff_10 = max_date - pd.Timedelta(days=10)
cutoff_20 = max_date - pd.Timedelta(days=20)
cutoff_30 = max_date - pd.Timedelta(days=30)

# Initialise counters
global_recent10_counter = Counter()
global_recent_counter = Counter()  # 20‑day recent global
global_recent30_counter = Counter()  # 30‑day recent global
overall_counter = Counter()  # overall global

cust_recent10_counters = defaultdict(Counter)  # per‑customer 10‑day recent
cust_recent_counters = defaultdict(Counter)  # per‑customer 20‑day recent
cust_recent30_counters = defaultdict(Counter)  # per‑customer 30‑day recent
cust_overall_counters = defaultdict(Counter)  # per‑customer overall

# Second pass: populate all counters
for chunk in pd.read_csv(
    train_path,
    usecols=["customer_id", "article_id", "t_dat"],
    dtype={"customer_id": str, "article_id": str, "t_dat": str},
    chunksize=chunksize,
):
    chunk["t_dat"] = pd.to_datetime(chunk["t_dat"])

    # overall counts
    overall_counter.update(chunk["article_id"])
    for cust_id, art_id in zip(chunk["customer_id"], chunk["article_id"]):
        cust_overall_counters[cust_id][art_id] += 1

    # 20‑day recent
    recent20 = chunk[chunk["t_dat"] >= cutoff_20]
    global_recent_counter.update(recent20["article_id"])
    for cust_id, art_id in zip(recent20["customer_id"], recent20["article_id"]):
        cust_recent_counters[cust_id][art_id] += 1

    # 30‑day recent
    recent30 = chunk[chunk["t_dat"] >= cutoff_30]
    global_recent30_counter.update(recent30["article_id"])
    for cust_id, art_id in zip(recent30["customer_id"], recent30["article_id"]):
        cust_recent30_counters[cust_id][art_id] += 1

    # 10‑day recent (new layer)
    recent10 = chunk[chunk["t_dat"] >= cutoff_10]
    global_recent10_counter.update(recent10["article_id"])
    for cust_id, art_id in zip(recent10["customer_id"], recent10["article_id"]):
        cust_recent10_counters[cust_id][art_id] += 1

# Top‑12 lists for each global layer
global_recent10_top12 = [art for art, _ in global_recent10_counter.most_common(12)]
global_recent_top12 = [art for art, _ in global_recent_counter.most_common(12)]
global_recent30_top12 = [art for art, _ in global_recent30_counter.most_common(12)]
global_overall_top12 = [art for art, _ in overall_counter.most_common(12)]

cust_pred_str = {}
all_cust_ids = (
    set(cust_recent10_counters.keys())
    | set(cust_recent_counters.keys())
    | set(cust_recent30_counters.keys())
    | set(cust_overall_counters.keys())
)

for cust_id in all_cust_ids:
    recent10_cnt = cust_recent10_counters.get(cust_id, Counter())
    recent20_cnt = cust_recent_counters.get(cust_id, Counter())
    recent30_cnt = cust_recent30_counters.get(cust_id, Counter())
    overall_cnt = cust_overall_counters.get(cust_id, Counter())

    top12 = [art for art, _ in recent10_cnt.most_common()]

    if len(top12) < 12:
        for art, _ in recent20_cnt.most_common():
            if art not in top12:
                top12.append(art)
                if len(top12) == 12:
                    break

    if len(top12) < 12:
        for art, _ in recent30_cnt.most_common():
            if art not in top12:
                top12.append(art)
                if len(top12) == 12:
                    break

    if len(top12) < 12:
        for art, _ in overall_cnt.most_common():
            if art not in top12:
                top12.append(art)
                if len(top12) == 12:
                    break

    if len(top12) < 12:
        for art in global_recent10_top12:
            if art not in top12:
                top12.append(art)
                if len(top12) == 12:
                    break

    if len(top12) < 12:
        for art in global_recent30_top12:
            if art not in top12:
                top12.append(art)
                if len(top12) == 12:
                    break

    if len(top12) < 12:
        for art in global_recent_top12:
            if art not in top12:
                top12.append(art)
                if len(top12) == 12:
                    break

    if len(top12) < 12:
        for art in global_overall_top12:
            if art not in top12:
                top12.append(art)
                if len(top12) == 12:
                    break

    cust_pred_str[cust_id] = " ".join(top12[:12])




In [3]:
def get_prediction(row):
    return cust_pred_str.get(row["customer_id"], " ".join(global_recent10_top12))


sample_sub["prediction"] = sample_sub.apply(get_prediction, axis=1)



In [4]:
output_path = "submission.csv"
sample_sub.to_csv(output_path, index=False)